# W5 Homework — Reflection on a Rendered Chart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week05/W5_hw_chart_reflection.ipynb)

**Goal.** Apply the reflection loop from the W5 lab — generate, execute, critique the
executed result, regenerate — to a harder output: matplotlib charts over a sales
dataset, with an LLM-judge score showing the revision beating the first draft.

In the lab the critique read text it could fully see. Here the draft is *code*, and
what actually needs judging is the *rendered chart* — so the critique receives the
image itself (vision input). Feedback grounded in the image can see defects the code
alone does not show (missing title, unreadable legend, overlapping labels). This is
the "new information enters the loop" case from lab Section 5. The pipeline:

1. **Generate (V1)** — a model call writes matplotlib code for a requested chart.
2. **Execute** — the code is extracted from the response and run, producing the chart.
3. **Reflect** — a vision-capable call reviews the rendered chart together with the
   original code and returns feedback plus revised code.
4. **Regenerate (V2)** — the revised code runs, producing the improved chart.

The path: setup (Section 1) → data (Section 2) → the pipeline step by step, with the
reflection prompt as the fill-in (Section 3) → the pipeline as one function
(Section 4) → the measured target — judge score V2 ≥ 4 and V2 > V1 — with two
exercises (Section 5) → completion (Section 6).

*Runtime:* Google Colab, top-to-bottom, ~60–80 minutes. Cells marked ✍️ ask for your own
writing — a fill-in or a written prediction. **Due before the W6 class.**

*Sources:* adapted from DeepLearning.AI, *Agentic AI* (Andrew Ng) — Module 2
(Reflection), ungraded labs 1–2. Workflow, step order, and prompt structure follow the
source; setup adapted to Colab and the course API standard.

## 1. Setup

Same setup as W1–W2: client library, key, helpers, one test call. Colab already
provides `pandas` and `matplotlib`.

### 1.1 Installation

`aisuite` exposes multiple providers behind one interface, so lab code stays identical
whichever provider your key belongs to.

*Do:* run the cell below (about 30 seconds, once per session).

In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

Paste your API key over `PASTE-YOUR-KEY-HERE` (issuing steps: the API Setup guide on
the course site). The reflection step sends the rendered chart image to the model, so
`VISION_MODEL` must accept image input; the default `gpt-4o-mini` does.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"          # Anthropic accounts: "anthropic:claude-haiku-4-5"
# The reflect step sends the rendered chart as an image, so VISION_MODEL must accept
# image input. Anthropic accounts: set BOTH strings to "anthropic:claude-haiku-4-5".
VISION_MODEL = "openai:gpt-4o-mini"

### 1.3 Client and helpers

`ask` sends a single text prompt; `ask_with_image` attaches a PNG file to the prompt
for the reflection step; `show_image` displays a saved chart inline. `n_calls` counts
API calls for the Section 5 exercises.

*Do:* run the cell unchanged.

In [ ]:
import base64
import json
import re

import aisuite
from IPython.display import Image, display

client = aisuite.Client()
n_calls = 0                            # API call counter, read in the Section 5 exercises


def ask(prompt, model=None, temperature=0.7):
    """Prompt string -> assistant reply string."""
    global n_calls
    n_calls += 1
    response = client.chat.completions.create(
        model=model or MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content


def ask_with_image(prompt, image_path, model=None):
    """(prompt, PNG path) -> assistant reply string, image attached to the message."""
    global n_calls
    n_calls += 1
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    response = client.chat.completions.create(
        model=model or VISION_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ],
        }],
    )
    return response.choices[0].message.content


def show_image(path):
    """PNG path -> inline display."""
    display(Image(filename=path))

### 1.4 Verification

*Do:* run the cell and confirm the output is exactly `ready`.

In [ ]:
print(ask("Reply with exactly: ready"))

If the output is `ready`, key and billing are correct. Any error at this point is a setup problem, not a code problem.

## 2. Data

The dataset is a coffee vending machine's sales log (one row per purchase: `date`,
`time`, payment fields, `price`, `coffee_name`). The cell downloads it from the course
repository; if the download fails, upload `coffee_sales.csv` manually through the Colab file
pane. Loading derives `year`, `quarter`, and `month` columns, which the chart prompts
refer to.

*Do:* run the cell and check the printed schema and row count.

In [ ]:
import urllib.request

import pandas as pd

CSV_URLS = [
    "https://raw.githubusercontent.com/ralbu85/stml_2026/main/labs/data/coffee_sales.csv",
    "https://stml.tailcce2b.ts.net/labs/data/coffee_sales.csv",
]
CSV_PATH = "coffee_sales.csv"

if not os.path.exists(CSV_PATH):
    for url in CSV_URLS:
        try:
            urllib.request.urlretrieve(url, CSV_PATH)
            break
        except OSError:
            continue


def load_and_prepare_data(csv_path):
    """CSV path -> DataFrame with derived year/quarter/month columns."""
    frame = pd.read_csv(csv_path)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame["quarter"] = frame["date"].dt.quarter
    frame["month"] = frame["date"].dt.month
    frame["year"] = frame["date"].dt.year
    return frame


df = load_and_prepare_data(CSV_PATH)
display(df.sample(n=5))

The chart requests in this lab aggregate these rows — for example, total Q1 sales per year. The model never sees the data itself; it sees only the column schema and writes code against it.

## 3. The Reflection Pipeline

### 3.1 First-draft code generation (V1)

The generation prompt follows the source lab and is given in full. It has three parts:
the **format contract** — return only Python code wrapped in `<execute_python>` tags,
assume `df` is loaded, use matplotlib, save to a given path, close the figure; the
**chart requirements** — title, axis labels, legend, correct aggregation; and the
column schema. Tag-wrapped output is what makes the next step mechanical — the same
contract W2 used with `<answer>` tags, and the convention W3 standardized into
function calling.

*Do:* run the cell; it assembles the generation prompt — nothing to edit.

In [ ]:
SCHEMA_TEXT = """\
Columns available in the DataFrame df:
- date (datetime)
- time (HH:MM)
- cash_type (card or cash)
- card (string)
- price (number)
- coffee_name (string)
- quarter (1-4)
- month (1-12)
- year (YYYY)"""

CHART_REQUIREMENTS = """\
- Add a clear, descriptive title.
- Label both axes.
- Add a legend when more than one series is plotted.
- Aggregate the data exactly as the instruction asks before plotting."""

CHART_FORMAT_CONTRACT = """\
You are a data visualization expert.

Return your answer *strictly* in this format:

<execute_python>
# valid python code here
</execute_python>

Do not add explanations, only the tags and the code.

Rules for the code:
1. Assume the DataFrame is already loaded as 'df'. Do not read any file.
2. Use matplotlib for plotting (no seaborn).
3. Save the figure as '{out_path}' with dpi=100.
4. Do not call plt.show().
5. Close all plots with plt.close().
6. Include all necessary import statements."""


def generate_chart_code(instruction, out_path, model=None):
    """(chart request, output PNG path) -> tag-wrapped matplotlib code."""
    prompt = (
        CHART_FORMAT_CONTRACT.format(out_path=out_path)
        + "\n\nChart requirements:\n" + CHART_REQUIREMENTS
        + "\n\n" + SCHEMA_TEXT
        + "\n\nUser instruction: " + instruction
    )
    return ask(prompt, model=model)

The instruction below is the one used in the source lab.

*Do:* run the cell and check the response is a single tagged code block.

In [ ]:
INSTRUCTION = ("Create a plot comparing Q1 coffee sales in 2024 and 2025 "
               "using the data in coffee_sales.csv.")

code_v1 = generate_chart_code(INSTRUCTION, "chart_v1.png")
print(code_v1)

### 3.2 Extraction and execution

A regular expression pulls the code out of the tags, and `exec` runs it with `df` in
scope. Executing model-written code is what produces the chart — and it is also the
step where the workflow hands control of the Python process to generated text, so
nothing here is sandboxed (notes Ch. 14 returns to this risk). In Colab the code runs inside
your own disposable VM.

*Do:* run the cell; the V1 chart appears below it.

In [ ]:
TAG_PATTERN = r"<execute_python>([\s\S]*?)</execute_python>"


def extract_code(tagged_response):
    """Tag-wrapped response -> bare code string ('' if no tags found)."""
    match = re.search(TAG_PATTERN, tagged_response)
    return match.group(1).strip() if match else ""


def run_chart_code(tagged_response, frame):
    """Extract the tagged code and execute it with the DataFrame in scope."""
    chart_code = extract_code(tagged_response)
    # exec runs unsandboxed model-written code; acceptable here only because the
    # notebook runs in a disposable Colab VM (→ notes Ch. 14, security).
    exec(chart_code, {"df": frame})


run_chart_code(code_v1, df)
show_image("chart_v1.png")

This is the first draft (V1). Depending on the run, it may lack a title, mislabel an axis, pick an unreadable layout, or slice the data incorrectly — first drafts vary. The reflection step exists precisely because a single generation call has no step that checks its own output.

### 3.3 Reflection on the rendered chart ✍️

The reflect step sends the model three things: the rendered chart image, the original
code, and the original instruction. Its output contract is fixed and has two parts —
first line, a JSON object with a single `feedback` field; then the revised code in
`<execute_python>` tags. The parsing code depends on that contract, so it is provided.

Write `REFLECTION_PROMPT`: the critique instruction placed at the top of the prompt.
It should tell the model what to check the chart against — whether it answers the
instruction, whether the data slice is correct, and whether title, axis labels,
legend, and tick labels are present and readable — and to produce improved code that
fixes what the critique found.

*Do:* fill in `REFLECTION_PROMPT` between the markers and run the cell.

In [ ]:
### FILL IN (START) ###
REFLECTION_PROMPT = (
    ""
)
### FILL IN (END) ###

REFLECTION_FORMAT_CONTRACT = """\
OUTPUT FORMAT (STRICT):
1) First line: a valid JSON object with ONLY the "feedback" field.
   Example: {{"feedback": "The legend is unclear and the axis labels overlap."}}
2) After a newline, output ONLY the refined Python code wrapped in:
<execute_python>
...
</execute_python>

Rules for the code:
- No markdown, backticks, or prose outside the two parts above.
- Use pandas/matplotlib only (no seaborn).
- Assume df already exists; do not read any file.
- Save the figure as '{out_path}' with dpi=100.
- Do not call plt.show(); close all plots with plt.close().
- Include all necessary import statements."""


def reflect_and_regenerate(chart_path, instruction, original_code, out_path,
                           model=None):
    """(chart PNG, instruction, V1 code, output path) -> (feedback, tagged V2 code)."""
    prompt = (
        REFLECTION_PROMPT
        + "\n\nOriginal code (for context):\n" + original_code
        + "\n\n" + SCHEMA_TEXT
        + "\n\nInstruction:\n" + instruction
        + "\n\n" + REFLECTION_FORMAT_CONTRACT.format(out_path=out_path)
    )
    content = ask_with_image(prompt, chart_path, model=model)

    lines = content.strip().splitlines()
    try:
        feedback = json.loads(lines[0].strip())["feedback"]
    except (json.JSONDecodeError, KeyError, IndexError):
        found = re.search(r"\{.*?\}", content, flags=re.DOTALL)
        feedback = (json.loads(found.group(0)).get("feedback", "")
                    if found else "")

    revised = extract_code(content)
    if revised:
        # extract_code strips the tags; re-wrap so V2 runs through the same path as V1.
        revised = f"<execute_python>\n{revised}\n</execute_python>"
    return str(feedback).strip(), revised

The call below critiques the V1 chart. The feedback names concrete defects; the
revised code should address them.

*Do:* run the cell and read the feedback text.

In [ ]:
feedback, code_v2 = reflect_and_regenerate(
    chart_path="chart_v1.png",
    instruction=INSTRUCTION,
    original_code=code_v1,
    out_path="chart_v2.png",
)

print("FEEDBACK:", feedback)
print()
print(code_v2)

### 3.4 Execution of the revision (V2)

The revised code runs through the same extract-and-execute step as V1.

*Do:* run the cell and compare the two charts side by side.

In [ ]:
run_chart_code(code_v2, df)

print("V1:")
show_image("chart_v1.png")
print("V2:")
show_image("chart_v2.png")

Read the two charts against the feedback text: each defect the critique named should be absent from V2. When V2 introduces a new defect instead, the feedback was too vague — Section 5 measures this, and sharpening `REFLECTION_PROMPT` is the fix.

## 4. End-to-End Workflow

The pipeline composes into one function, mirroring the source lab's `run_workflow`:

1. **Load** — read the CSV and derive date columns.
2. **Generate V1** — `generate_chart_code` writes the first-draft code.
3. **Execute V1** — extract and run, producing `<basename>_v1.png`.
4. **Reflect** — `reflect_and_regenerate` critiques the rendered chart and returns
   feedback plus revised code.
5. **Execute V2** — extract and run, producing `<basename>_v2.png`.

The return value keeps every intermediate artifact, because inspecting intermediates
is how a workflow gets debugged.

*Do:* run the cell; it defines `run_workflow` — nothing to edit.

In [ ]:
def run_workflow(csv_path, instruction, generation_model=None,
                 reflection_model=None, image_basename="chart"):
    """(CSV, chart request) -> dict with codes, feedback, and both chart paths."""
    frame = load_and_prepare_data(csv_path)
    out_v1 = f"{image_basename}_v1.png"
    out_v2 = f"{image_basename}_v2.png"

    draft = generate_chart_code(instruction, out_v1, model=generation_model)
    run_chart_code(draft, frame)

    critique, revision = reflect_and_regenerate(
        chart_path=out_v1, instruction=instruction,
        original_code=draft, out_path=out_v2, model=reflection_model,
    )
    run_chart_code(revision, frame)

    return {"code_v1": draft, "chart_v1": out_v1, "feedback": critique,
            "code_v2": revision, "chart_v2": out_v2}

### 4.1 A full run

The run below uses a distinct `image_basename` so its files do not overwrite the
Section 3 charts.

*Do:* run the cell. After the measured task in Section 5, rerun it with your own chart
instruction — any aggregation the schema supports works (sales by hour of day, by
coffee type, by payment method).

In [ ]:
result = run_workflow(
    csv_path="coffee_sales.csv",
    instruction=INSTRUCTION,
    generation_model=MODEL,
    reflection_model=VISION_MODEL,
    image_basename="drink_sales",
)

print("FEEDBACK:", result["feedback"])
show_image(result["chart_v1"])
show_image(result["chart_v2"])

## 5. Improvement with a Number ✍️ (core)

The claim "V2 is better than V1" becomes a measurement. **LLM-as-judge** = using a
model call to grade an output against a stated criterion. The judge receives the
instruction and the plotting code and awards one point per criterion:

| Criterion | Awarded when the code... |
|---|---|
| `data_slice` | selects exactly the data the instruction asks for |
| `title` | sets a descriptive title |
| `axis_labels` | labels both axes |
| `legend_readability` | adds a legend when several series are plotted, with readable labels |
| `matches_instruction` | produces the chart type and comparison the instruction describes |

Target: V2 total ≥ 4, and V2 > V1. If the target is missed, sharpen
`REFLECTION_PROMPT` and rerun Section 4 — the critique prompt is the tunable part of
this pipeline.

*Do:* run the collapsed judge cell as-is; on a miss, sharpen `REFLECTION_PROMPT` and
rerun Section 4.

In [ ]:
#@title Judge code — run as-is (applies the criteria table above) { display-mode: "form" }
CRITERIA = {
    "data_slice": "selects exactly the data the instruction asks for",
    "title": "sets a descriptive title",
    "axis_labels": "labels both axes",
    "legend_readability": "adds a legend when several series are plotted, with readable labels",
    "matches_instruction": "produces the chart type and comparison the instruction describes",
}

TARGET_SCORE = 4


def judge_chart(tagged_code, instruction):
    """(tagged plotting code, instruction) -> {criterion: 0/1} dict."""
    criteria_text = "\n".join(f'- "{name}": 1 if the code {desc}, else 0'
                              for name, desc in CRITERIA.items())
    prompt = (
        "You grade matplotlib plotting code against a chart instruction.\n"
        "Return a JSON object only, no prose, with these integer fields:\n"
        + criteria_text
        + "\n\nInstruction:\n" + instruction
        + "\n\nCode:\n" + extract_code(tagged_code)
    )
    reply = ask(prompt, temperature=0.0)
    found = re.search(r"\{[\s\S]*\}", reply)
    scores = json.loads(found.group(0)) if found else {}
    return {name: int(scores.get(name, 0)) for name in CRITERIA}


score_v1 = judge_chart(result["code_v1"], INSTRUCTION)
score_v2 = judge_chart(result["code_v2"], INSTRUCTION)
total_v1, total_v2 = sum(score_v1.values()), sum(score_v2.values())

print("V1:", score_v1, "->", total_v1)
print("V2:", score_v2, "->", total_v2)
print("target met:", total_v2 >= TARGET_SCORE and total_v2 > total_v1)

The per-criterion dict shows where the gain came from — typically `title`, `axis_labels`, and `legend_readability`, the defects a rendered image makes visible. A judge scoring code (not the image) can miss purely visual defects such as overlapping ticks; the exercise below probes exactly that gap.

### Exercise — a second reflection round ✍️

A third version (V3) costs two more API calls (one reflect, one judge). Prediction,
written down first: does V3's score improve on V2 by enough to justify the added
calls, or has the score saturated?

*Do:* run the cell and compare with your prediction.

In [ ]:
calls_before = n_calls

feedback_3, code_v3 = reflect_and_regenerate(
    chart_path=result["chart_v2"],
    instruction=INSTRUCTION,
    original_code=result["code_v2"],
    out_path="chart_v3.png",
)
run_chart_code(code_v3, df)

score_v3 = judge_chart(code_v3, INSTRUCTION)
print("V1 ->", total_v1, "| V2 ->", total_v2, "| V3 ->", sum(score_v3.values()))
print("extra calls for V3:", n_calls - calls_before)

On a 5-point checklist the second round usually adds little: reflection has diminishing returns, and each round multiplies cost. Production reflection loops therefore set a fixed iteration budget or a stop condition (score reached, feedback empty) rather than reflecting until perfect.

### Exercise — feedback source: code-only vs. rendered chart ✍️

Adapted from the source course's second reflection lab (SQL refinement), where a
reviewer that saw only the SQL text approved a query whose executed output was visibly
wrong; only feeding the execution result back exposed the bug. The analog here:
critique with and without the rendered image.

The variant below sends the identical reflection prompt with no image attached — the
model reviews the code alone, never the executed result. Prediction, written down
first: which critique catches more real defects, and which score is higher?

*Do:* run the cell and compare with your prediction.

In [ ]:
def reflect_code_only(instruction, original_code, out_path, model=None):
    """Reflection variant: identical prompt, no chart image attached."""
    prompt = (
        REFLECTION_PROMPT
        + "\n\nOriginal code (for context):\n" + original_code
        + "\n\n" + SCHEMA_TEXT
        + "\n\nInstruction:\n" + instruction
        + "\n\n" + REFLECTION_FORMAT_CONTRACT.format(out_path=out_path)
    )
    content = ask(prompt, model=model)
    lines = content.strip().splitlines()
    try:
        fb = json.loads(lines[0].strip())["feedback"]
    except (json.JSONDecodeError, KeyError, IndexError):
        found = re.search(r"\{.*?\}", content, flags=re.DOTALL)
        fb = json.loads(found.group(0)).get("feedback", "") if found else ""
    return str(fb).strip(), extract_code(content)


feedback_text_only, code_text_only = reflect_code_only(
    INSTRUCTION, result["code_v1"], "chart_v2_codeonly.png")

print("image-based feedback :", result["feedback"])
print("code-only feedback   :", feedback_text_only)

score_code_only = judge_chart(
    f"<execute_python>\n{code_text_only}\n</execute_python>", INSTRUCTION)
print("code-only V2 ->", sum(score_code_only.values()),
      "| image-based V2 ->", total_v2)

The code-only critique can verify the code against the instruction, but it cannot see rendering defects — overlapping labels, unreadable colors, a legend covering the data. The executed output is feedback that the code text does not contain; reflection improves the most when it is grounded in such external evidence, and degrades toward guessing when it reviews only its own text.

## 6. Completion Check

Completion criteria: the reflection prompt written, both pipeline charts produced, and
the Section 5 target reached. The checks are structural — content quality is graded by
the Section 5 score, not here. All rows must read `PASS` before submission.

In [ ]:
checks = {
    "REFLECTION_PROMPT written":
        len(REFLECTION_PROMPT.strip()) > 20,
    "workflow produced both charts":
        os.path.exists(result["chart_v1"]) and os.path.exists(result["chart_v2"]),
    "judge returned all criteria":
        set(score_v2) == set(CRITERIA),
    "target met (V2 >= 4 and V2 > V1)":
        total_v2 >= TARGET_SCORE and total_v2 > total_v1,
}

for name, passed in checks.items():
    print(("PASS " if passed else "FAIL "), name)
print("\nHOMEWORK COMPLETE" if all(checks.values()) else "\nNOT COMPLETE YET")

---

Reference fill-ins: `labs/checkpoints/week05/solution.py`, published after the
homework deadline.

Next week splits the work across several agents (notes Ch. 6) — and the
generate → execute → critique → regenerate cycle you ran here returns there with the
critic as a separate agent: the evaluator–optimizer pattern.